In [1]:
import os  # Impordib operatsioonisüsteemi mooduli keskkonnamuutujate lugemiseks
import pandas as pd  # Impordib Pandas teegi andmetöötluseks (lühendiga pd)
from supabase import create_client  # Impordib Supabase'i kliendi loomise funktsiooni
from dotenv import load_dotenv  # Impordib funktsiooni .env failist muutujate laadimiseks
import plotly.express as px  # Impordib Plotly teegi graafikute joonistamiseks (lühendiga px)

load_dotenv()  # Loeb .env failis olevad muutujad (URL ja KEY) mällu

supabase = create_client(  # Loo Supabase'i ühenduse klient
    os.getenv('SUPABASE_URL'),  # Loe keskkonnamuutujast Supabase'i projekti URL
    os.getenv('SUPABASE_KEY')   # Loe keskkonnamuutujast Supabase'i API võti
)

def get_data(tabel_name):  # Defineeri funktsioon get_data, mis võtab sisendiks tabeli nime
  data = []  # Loo tühi list kõigi lehekülgede andmete kogumiseks
  page_size = 1000  # Määra, et korraga küsitakse 1000 rida (Supabase limiit)
  page = 0  # Määra algseks lehekülje indeksiks 0
  while True:  # Korda päringut tsüklis, kuni kõik andmed on kätte saadud
    response = supabase.table(tabel_name).select('*').range(page * page_size, (page + 1) * page_size - 1).execute()  # Küsi Supabase'ist lehekülg andmeid (nt vahemik 0-999, siis 1000-1999)
    data.extend(response.data)  # Lisa saadud read olemasolevasse 'data' listi
    if len(response.data) < page_size:  # Kui saadud ridade arv on väiksem kui 1000, tähendab see viimast lehte
      break  # Katkesta tsükkel
    page += 1  # Suurenda lehekülje numbrit järgmise päringu jaoks
  return pd.DataFrame(data)  # Teisenda kogutud andmete list Pandase DataFrame'iks ja tagasta see

# response = supabase.table('sales').select('*').execute()  # (Kommenteeritud) Ühekordne liiga lihtne päring ilma lehitsemiseta
# df = pd.DataFrame(response.data)  # (Kommenteeritud) Teisendamine DataFrame'iks

df_sales = get_data('sales')  # Tõmba Supabase'ist sisse kogu 'sales' tabel DataFrame'ina
df_customers = get_data('customers')  # Tõmba Supabase'ist sisse kogu 'customers' tabel DataFrame'ina
df_products = get_data('products')  # Tõmba Supabase'ist sisse kogu 'products' tabel DataFrame'ina

df = pd.merge(df_sales, df_customers, on='customer_id', how='left')  # Ühenda müügid ja kliendid customer_id alusel
df = pd.merge(df, df_products, on='product_id', how='left')  # Ühenda saadud tabel toodetega product_id alusel

print(df.shape)  # Prindi ekraanile ühendatud tabeli mõõtmed (ridade arv, tulpade arv)

city_revenue = df.groupby('store_location')['total_price'].sum()  # Grupeeri poe asukoha järgi ja arvuta kogukäive (total_price sum)
print("Revenue by City:")  # Prindi pealkiri
print(city_revenue)  # Prindi linnade kaupa arvutatud käived

print("Number of Customers:", df['customer_id'].nunique())  # Arvuta ja prindi unikaalsete klientide koguarv

# 1. Leia TOP 5 toodet kogukäibe järgi
top_products = df.groupby('product_name')['total_price'].sum().nlargest(5)  # Grupeeri tootenime järgi, arvuta käive ja võta 5 suurimat
print(top_products)  # Prindi TOP 5 toodet

# 2. Leia klient suurima kogukulutusega
top_customer = df.groupby('customer_id')['total_price'].sum().idxmax()  # Grupeeri kliendi järgi, arvuta kulutused ja leia suurima summaga kliendi ID
print("Top customer:", top_customer)  # Prindi parima kliendi ID

df['sale_date'] = pd.to_datetime(df['sale_date'])  # Teisenda sale_date tulp teksti kujult kuupäeva tüüpi (datetime)

sales_over_time = (df.groupby('sale_date')['total_price'].sum().reset_index())  # Grupeeri kuupäeva järgi, arvuta päevakäive ja muuda tulemus tavaliseks tabeliks
sales_over_time.columns = ['Müügikuupäev', 'Müük']  # Nimeta tabeli tulbad uuesti eestikeelseks

fig_line = px.line(  # Loo Plotly joongraafiku objekt
    sales_over_time,  # Määra andmeallikaks sales_over_time tabel
    x='Müügikuupäev',  # X-teljele pane müügikuupäev
    y='Müük',  # Y-teljele pane müügisumma
    title='Müük läbi aegade',  # Määra graafikule pealkiri
    markers=True  # Lisa joonel olevatele andmepunktidele täpid (märgised)
)

fig_line.update_layout(  # Uuenda graafiku kujundust
    xaxis_title='Kuupäev',  # Määra X-telje silt
    yaxis_title='Müük (€)'  # Määra Y-telje silt
)
fig_line.show()  # Kuva valmis graafik ekraanile

(10118, 28)
Revenue by City:
store_location
Pärnu       288744.04
Tallinn    1092083.15
Tartu       521603.11
Name: total_price, dtype: float64
Number of Customers: 2551
product_name
Õhuline sünteetiline sporditossud    27347.04
Trendikas goretex oxfordid           23376.15
Praktiline viskoosne jakk            22188.80
Praktiline džersii seelik            22039.98
Boheemlaslik puuvillane tuulejope    21309.96
Name: total_price, dtype: float64
Top customer: 3618.0


In [2]:
#Iseseisev töö 1A

import pandas as pd

data = {
    'customer_id': [1001, 1002, 1003, 1001, 1002, 1004, 1003, 1001, 1005, 1004,
                    1002, 1003, 1005, 1001, 1006, 1004, 1002, 1007, 1003, 1005],
    'sale_date': ['2024-01-15', '2024-01-16', '2024-02-01', '2024-02-20', '2024-03-01',
                  '2024-03-05', '2024-03-15', '2024-04-10', '2024-04-12', '2024-04-20',
                  '2024-05-01', '2024-05-10', '2024-05-15', '2024-06-01', '2024-06-05',
                  '2024-06-10', '2024-06-20', '2024-07-01', '2024-07-05', '2024-07-10'],
    'total_price': [89.99, 45.50, 120.00, 67.30, 55.00, 210.00, 33.50, 145.00, 78.00, 92.00,
                    160.00, 44.00, 88.50, 230.00, 37.00, 175.00, 110.00, 65.00, 95.00, 125.00],
    'store_location': ['Tallinn', 'Tartu', 'Tallinn', 'Tallinn', 'Tartu', 'Pärnu', 'Tallinn', 'Tallinn',
            'Tartu', 'Pärnu', 'Tartu', 'Tallinn', 'Tartu', 'Tallinn', 'Pärnu', 'Pärnu',
            'Tartu', 'Tallinn', 'Tallinn', 'Tartu'],
    'product_category': ['Dresses', 'Tops', 'Denim', 'Accessories', 'Tops', 'Denim', 'Tops',
                        'Dresses', 'Denim', 'Accessories', 'Dresses', 'Tops', 'Denim',
                        'Dresses', 'Accessories', 'Denim', 'Tops', 'Accessories', 'Dresses', 'Denim']
}

df = pd.DataFrame(data)

print("=== SHAPE ===")
print(df.shape)

print("\n=== HEAD ===")
print(df.head())

print("\n=== INFO ===")
print(df.info())

print("\n=== DESCRIBE ===")
print(df.describe())

print("\n=== DTYPES ===")
print(df.dtypes)

=== SHAPE ===
(20, 5)

=== HEAD ===
   customer_id   sale_date  total_price store_location product_category
0         1001  2024-01-15        89.99        Tallinn          Dresses
1         1002  2024-01-16        45.50          Tartu             Tops
2         1003  2024-02-01       120.00        Tallinn            Denim
3         1001  2024-02-20        67.30        Tallinn      Accessories
4         1002  2024-03-01        55.00          Tartu             Tops

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       20 non-null     int64  
 1   sale_date         20 non-null     str    
 2   total_price       20 non-null     float64
 3   store_location    20 non-null     str    
 4   product_category  20 non-null     str    
dtypes: float64(1), int64(1), str(3)
memory usage: 1.4 KB
None

=== DESCRIBE ===
       customer_i

In [3]:
# Iseseisev töö 1B

# 1. Mitu unikaalset klienti on andmestikus?
print("Unikaalseid kliente:", 
      df['customer_id'].nunique())

# 2. Millistes asukohtades on meie kauplused?
print("Asukohad:", 
      df['store_location'].unique())

# 3. Mitu tellimust on igast asukohast?
print("Tellimused asukoha järgi:\n", 
      df['store_location'].value_counts())

# 4. Mis on kogu käive?
print("Kogukäive:", 
      df['total_price'].sum())

Unikaalseid kliente: 7
Asukohad: <ArrowStringArray>
['Tallinn', 'Tartu', 'Pärnu']
Length: 3, dtype: str
Tellimused asukoha järgi:
 store_location
Tallinn    9
Tartu      7
Pärnu      4
Name: count, dtype: int64
Kogukäive: 2065.79


In [4]:
#iseseisev töö 1C
# Uurime tootekategooriaid
print("1. Unikaalsed kategooriad:", 
      df['product_category'].unique())
print("\n2. Mitu tellimust kategooria kohta:\n", 
      df['product_category'].value_counts())
print("\n3. Käive kategooriate kaupa:\n", 
      df.groupby('product_category')['total_price'].sum())


1. Unikaalsed kategooriad: <ArrowStringArray>
['Dresses', 'Tops', 'Denim', 'Accessories']
Length: 4, dtype: str

2. Mitu tellimust kategooria kohta:
 product_category
Denim          6
Dresses        5
Tops           5
Accessories    4
Name: count, dtype: int64

3. Käive kategooriate kaupa:
 product_category
Accessories    261.30
Denim          796.50
Dresses        719.99
Tops           288.00
Name: total_price, dtype: float64


In [5]:
#Iseseisev töö 2A

import pandas as pd

# 1. Filtreerimine (WHERE)
print("=== Tallinna tellimused ===")
tallinn = df[df['store_location'] == 'Tallinn']
print(f"Kokku: {len(tallinn)} tellimust, käive: {tallinn['total_price'].sum():.2f} EUR")

# 2. Grupeerimine (GROUP BY)
print("\n=== Käive linniti ===")
city_revenue = df.groupby('store_location')['total_price'].agg(['sum', 'mean', 'count'])
city_revenue.columns = ['kogukäive', 'keskmine', 'tellimusi']
print(city_revenue.sort_values('kogukäive', ascending=False))

# 3. Uue veeru loomine (CASE WHEN)
df['order_size'] = df['total_price'].apply(
    lambda x: 'Suur (100+)' if x >= 100 else 'Väike (<100)'
)
print("\n=== Tellimused suuruse järgi ===")
print(df['order_size'].value_counts())

=== Tallinna tellimused ===
Kokku: 9 tellimust, käive: 889.79 EUR

=== Käive linniti ===
                kogukäive    keskmine  tellimusi
store_location                                  
Tallinn            889.79   98.865556          9
Tartu              662.00   94.571429          7
Pärnu              514.00  128.500000          4

=== Tellimused suuruse järgi ===
order_size
Väike (<100)    12
Suur (100+)      8
Name: count, dtype: int64


In [6]:
#Iseseisev töö 2B
# Klientide kokkuvõte
customer_summary = df.groupby('customer_id')['total_price'].agg(
    kogukulutus='sum',
    keskmine='mean',
    tellimusi='count'
).reset_index()

# Sorteeri
customer_summary = customer_summary.sort_values('kogukulutus', ascending=False)

# Lisa VIP staatus (täidetud lüngad)
customer_summary['vip_status'] = customer_summary['kogukulutus'].apply(
    lambda x: 'VIP' if x > 200 else 'Tavaline'
)

print(customer_summary)

   customer_id  kogukulutus    keskmine  tellimusi vip_status
0         1001       532.29  133.072500          4        VIP
3         1004       477.00  159.000000          3        VIP
1         1002       370.50   92.625000          4        VIP
2         1003       292.50   73.125000          4        VIP
4         1005       291.50   97.166667          3        VIP
6         1007        65.00   65.000000          1   Tavaline
5         1006        37.00   37.000000          1   Tavaline


In [7]:
# Iseseisev töö 2C
# Kliendiandmed
customers = pd.DataFrame({
    'customer_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007],
    'first_name': ['Jüri', 'Kati', 'Maris', 'Peeter', 'Liina', 'Andres', 'Tiina'],
    'email': ['juri@mail.ee', 'kati@mail.ee', 'maris@mail.ee', 'peeter@mail.ee',
              'liina@mail.ee', 'andres@mail.ee', 'tiina@mail.ee']
})

# Ühenda müügiandmetega
merged = pd.merge(df, customers, on='customer_id', how='left')

# Arvuta TOP 5 klienti nime järgi
top_5_customers = merged.groupby(['customer_id', 'first_name'])['total_price'].sum().reset_index()
top_5_customers = top_5_customers.sort_values('total_price', ascending=False).head(5)

print("=== TOP 5 KLIENDI KÄIVE ===")
print(top_5_customers)

=== TOP 5 KLIENDI KÄIVE ===
   customer_id first_name  total_price
0         1001       Jüri       532.29
3         1004     Peeter       477.00
1         1002       Kati       370.50
2         1003      Maris       292.50
4         1005      Liina       291.50


In [8]:
#Iseseisev töö 3A
import pandas as pd
import plotly.express as px

# Käive kategooriate kaupa
cat_revenue = df.groupby('product_category')['total_price'].sum().reset_index()
cat_revenue = cat_revenue.sort_values('total_price', ascending=True)

fig = px.bar(
    cat_revenue,
    x='total_price',
    y='product_category',
    orientation='h',
    title='UrbanStyle: Müügikäive kategooriate kaupa (2024)',
    labels={
        'total_price': 'Käive (EUR)',
        'product_category': 'Kategooria'
    },
    color='product_category',
    text='total_price'  # Kuvab täpse summa tulpa peal
)

# Üleliigse legendi peitmine (kuna Y-teljel on kategooriad juba kirjas)
fig.update_layout(showlegend=False)

fig.show()


In [9]:
 #Iseseisev töö 3B - kuukäive trend
# Teisendame kuupäeva ja grupeerime kuu kaupa
df['sale_date'] = pd.to_datetime(df['sale_date'])
df['month'] = df['sale_date'].dt.strftime('%Y-%m')

monthly_revenue = df.groupby('month')['total_price'].sum().reset_index()

fig_line = px.line(
    monthly_revenue,
    x='month',
    y='total_price',
    title='UrbanStyle: Kuukäive trend (2024)',
    labels={'month': 'Kuu', 'total_price': 'Käive (€)'},
    markers=True
)
fig_line.show()


In [10]:
import plotly.express as px

# 1. Arvutame kliendi kokkuvõtte ja VIP staatuse uuesti (et ei sõltuks eelmistest lahtritest)
customer_summary = df.groupby('customer_id')['total_price'].agg(
    kogukulutus='sum',
    keskmine='mean',
    tellimusi='count'
).reset_index()

customer_summary['vip_status'] = customer_summary['kogukulutus'].apply(
    lambda x: 'VIP' if x > 200 else 'Tavaline'
)

# 2. Joonistame sektordiagrammi (Graafik 2)
fig_pie = px.pie(
    customer_summary,
    names='vip_status',
    title='Klientide jaotus VIP-staatuse järgi',
    color='vip_status',
    color_discrete_map={'VIP': '#FFD700', 'Tavaline': '#87CEEB'}
)

fig_pie.show()

In [11]:
import pandas as pd
import plotly.express as px

# Samm 1 — Andmete laadimine ja kuupäevad
df['sale_date'] = pd.to_datetime(df['sale_date'])
today = pd.to_datetime('2024-08-01')

# Samm 2 — RFM näitajate arvutamine
recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase']
recency['recency_days'] = (today - recency['last_purchase']).dt.days

frequency = df.groupby('customer_id').size().reset_index(name='frequency')

monetary = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary.columns = ['customer_id', 'monetary']

rfm = recency[['customer_id', 'recency_days']].merge(
    frequency, on='customer_id'
).merge(
    monetary, on='customer_id'
)

# Samm 3 — Skooride määramine (R_score, F_score, M_score)
rfm['R_score'] = pd.qcut(rfm['recency_days'], q=3, labels=[3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=3, labels=[1, 2, 3]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'], q=3, labels=[1, 2, 3]).astype(int)

# Koguskoor ja segmendid
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

def assign_segment(score):
    if score >= 8:
        return 'VIP Champions'
    elif score >= 6:
        return 'Loyal Customers'
    elif score >= 4:
        return 'Potential Loyalists'
    else:
        return 'At Risk'

rfm['segment'] = rfm['RFM_score'].apply(assign_segment)

print("=== RFM TULEMUSED ===")
print(rfm.sort_values('RFM_score', ascending=False))

# Samm 4 — Visualiseerimine
# Graafik 1: Segmentide jaotus
segment_counts = rfm['segment'].value_counts().reset_index()
fig1 = px.bar(
    segment_counts,
    x='segment',
    y='count',
    title='UrbanStyle: Kliendisegmentide jaotus (RFM)',
    labels={'segment': 'Segment', 'count': 'Klientide arv'},
    color='segment'
)
fig1.show()

# Graafik 2: Recency vs Monetary hajuvusdiagramm
fig2 = px.scatter(
    rfm,
    x='recency_days',
    y='monetary',
    color='segment',
    size='frequency',
    hover_data=['customer_id'],
    title='UrbanStyle: Recency vs Monetary (RFM)',
    labels={
        'recency_days': 'Päevi viimasest ostust (väiksem on parem)',
        'monetary': 'Kogukulutus (EUR)'
    }
)
fig2.show()

=== RFM TULEMUSED ===
   customer_id  recency_days  frequency  monetary  R_score  F_score  M_score  \
2         1003            27          4    292.50        3        3        2   
1         1002            42          4    370.50        2        3        2   
0         1001            61          4    532.29        1        2        3   
3         1004            52          3    477.00        2        1        3   
4         1005            22          3    291.50        3        2        1   
6         1007            31          1     65.00        3        1        1   
5         1006            57          1     37.00        1        1        1   

   RFM_score              segment  
2          8        VIP Champions  
1          7      Loyal Customers  
0          6      Loyal Customers  
3          6      Loyal Customers  
4          6      Loyal Customers  
6          5  Potential Loyalists  
5          3              At Risk  
